# Baseline Convolutional Neural Network

In [ ]:
pip install torch torchvision torchaudio pandas numpy matplotlib scikit-learn wandb

# Colab & Kaggle Setup
Run this section only in Google Colab. It downloads the FER2013 dataset directly from Kaggle into the Colab environment.

In [ ]:
from google.colab import files

# Download kaggle.json from kaggle.com → top-right avatar → Settings → API → Create New Token
files.upload()  # select the kaggle.json file from your computer

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle competitions download -c challenges-in-representation-learning-facial-expression-recognition-challenge
!unzip -q challenges-in-representation-learning-facial-expression-recognition-challenge.zip -d data/

# WandB Login

In [ ]:
wandb.login()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

import wandb

In [10]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

In [ ]:
DATA_DIR = "data"  # Colab path after Kaggle download — change to "../data" if running locally

train_df = pd.read_csv(f"{DATA_DIR}/train.csv")

print(train_df.shape)
train_df.head()

# Train/Validaation Split

In [12]:
train_data, val_data = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42,
    stratify=train_df["emotion"]
)

print("Train:", train_data.shape)
print("Validation:", val_data.shape)

Train: (22967, 2)
Validation: (5742, 2)


In [13]:
class FERDataset(Dataset):
    def __init__(self, dataframe):
        self.dataframe = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        pixels = np.array(row["pixels"].split(), dtype=np.float32)
        image = pixels.reshape(48, 48)
        image = image / 255.0

        image = torch.tensor(image, dtype=torch.float32).unsqueeze(0)

        label = int(row["emotion"])
        label = torch.tensor(label, dtype=torch.long)

        return image, label

In [14]:
train_dataset = FERDataset(train_data)
val_dataset = FERDataset(val_data)

image, label = train_dataset[0]

print("Image shape:", image.shape)
print("Label:", label)
print("Min pixel:", image.min().item())
print("Max pixel:", image.max().item())

Image shape: torch.Size([1, 48, 48])
Label: tensor(6)
Min pixel: 0.027450980618596077
Max pixel: 0.8745098114013672


# Create DataLoaders

In [15]:
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False
)

In [16]:
images, labels = next(iter(train_loader))

print(images.shape)
print(labels.shape)

torch.Size([64, 1, 48, 48])
torch.Size([64])


# BaselineCNN

In [17]:
class BaselineCNN(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
        
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 12 * 12, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes),
        )
    
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x
    

In [18]:
model = BaselineCNN().to(device)

images, labels = next(iter(train_loader))
images = images.to(device)

outputs = model(images)

print("Input shape:", images.shape)
print("Output shape:", outputs.shape)

Input shape: torch.Size([64, 1, 48, 48])
Output shape: torch.Size([64, 7])


In [19]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

images, labels = next(iter(train_loader))

images = images.to(device)
labels = labels.to(device)

outputs = model(images)
loss = criterion(outputs, labels)

optimizer.zero_grad()
loss.backward()
optimizer.step()

print("Loss:", loss.item())

Loss: 1.9484626054763794


# WandB Setup

In [ ]:
run = wandb.init(
    entity="nestor_dzadzamia-free-university-of-tbilisi-",
    project="facial-expression-recognition",
    name="baseline-cnn",
    config={
        "architecture": "BaselineCNN",
        "dataset": "FER2013",
        "epochs": 20,
        "batch_size": 64,
        "learning_rate": 1e-3,
        "optimizer": "Adam",
        "num_conv_layers": 2,
        "filters": "16->32",
        "fc_units": 128,
    }
)

# Training Loop

In [ ]:
NUM_EPOCHS = run.config.epochs

model = BaselineCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=run.config.learning_rate)

for epoch in range(NUM_EPOCHS):

    # Train
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        train_correct += (preds == labels).sum().item()
        train_total += images.size(0)

    train_loss /= train_total
    train_acc = train_correct / train_total

    # Validate
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            val_correct += (preds == labels).sum().item()
            val_total += images.size(0)

    val_loss /= val_total
    val_acc = val_correct / val_total

    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc,
    })

In [ ]:
wandb.finish()